# SmartHire — 01. Exploratory Data Analysis (EDA)
This notebook conducts comprehensive exploratory data analysis on:
1. **Resume Dataset** (`data/raw/UpdatedResumeDataSet.csv`): 962 resumes across 25 job categories.
2. **Naukri Job Listings Dataset** (`data/raw/naukri_com-job_sample.csv`): 22,000 job postings with skills, requirements, and locations.

We examine distributions, class balance, missing data patterns, word counts, and technical skill frequencies using classical data science techniques.


In [ ]:
import sys
from pathlib import Path

# Relative repository setup
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import NAUKRI_RAW_PATH, RESUME_RAW_PATH, RESUMES_PROCESSED_PATH, JOBS_PROCESSED_PATH
from src.features.text_features import clean_text, extract_skills

print("Configuration loaded. Project root:", PROJECT_ROOT)


## 1. Resume Dataset Inspection & Class Balance


In [ ]:
df_resumes = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "UpdatedResumeDataSet.csv")
print("Resume dataset shape:", df_resumes.shape)
print("Columns:", df_resumes.columns.tolist())
display(df_resumes.head(3))


In [ ]:
# Check missing values and duplicate rows
print("Missing values in resumes:\n", df_resumes.isnull().sum())
print(f"Total rows: {len(df_resumes)}, Unique resumes: {df_resumes['Resume'].nunique()}")
print(f"Unique job categories: {df_resumes['Category'].nunique()}")


In [ ]:
# Class distribution visualization
category_counts = df_resumes['Category'].value_counts()
plt.figure(figsize=(14, 7))
sns.barplot(x=category_counts.values, y=category_counts.index, palette="viridis")
plt.title("Resume Distribution Across 25 Job Categories", fontsize=14, fontweight="bold")
plt.xlabel("Number of Resumes", fontsize=12)
plt.ylabel("Job Category", fontsize=12)
plt.tight_layout()
plt.show()


## 2. Resume Text Length & Word Count Statistics


In [ ]:
df_resumes['word_count'] = df_resumes['Resume'].apply(lambda x: len(str(x).split()))
df_resumes['char_count'] = df_resumes['Resume'].apply(lambda x: len(str(x)))

print("Resume Word Count Summary:")
display(df_resumes[['word_count', 'char_count']].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df_resumes['word_count'], bins=30, kde=True, color='teal')
plt.title("Distribution of Resume Word Lengths", fontsize=13)
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 3. Naukri Job Corpus Inspection & Missing Values


In [ ]:
df_jobs = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "naukri_com-job_sample.csv", low_memory=False)
print("Jobs raw shape:", df_jobs.shape)
print("Missing values in raw jobs:\n", df_jobs.isnull().sum())


In [ ]:
# Inspect top job titles and top hiring companies
top_titles = df_jobs['jobtitle'].dropna().value_counts().head(10)
print("Top 10 Job Titles in Corpus:\n", top_titles)

top_companies = df_jobs['company'].dropna().value_counts().head(10)
print("\nTop 10 Companies Posting Vacancies:\n", top_companies)


## 4. Skills Extraction Frequency in Job Corpus


In [ ]:
# Extract skills from a sample of 2,000 job descriptions
sample_descriptions = df_jobs['jobdescription'].dropna().sample(n=2000, random_state=42)
all_extracted_skills = []
for desc in sample_descriptions:
    skills = extract_skills(desc)
    all_extracted_skills.extend(skills)

skill_freq = pd.Series(all_extracted_skills).value_counts().head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x=skill_freq.values, y=skill_freq.index, palette="mako")
plt.title("Top 20 Most In-Demand Skills in Job Market Sample", fontsize=14, fontweight="bold")
plt.xlabel("Frequency Across Postings")
plt.ylabel("Normalized Skill")
plt.tight_layout()
plt.show()


## Key Insights & Next Steps
- The Resume dataset contains 962 records distributed across 25 job domains.
- Job descriptions contain rich technical skills with significant frequencies in Python, Java, SQL, Machine Learning, AWS, and Web technologies.
- Preprocessed datasets are exported to `data/processed/` for model training.
